Experiment B
Title

Adaptive Routing Optimization for Hybrid IDS

Research Question

Which confidence threshold provides the best routing strategy between Random Forest and OCSVM for detecting unknown attacks?

Hypothesis

An adaptive confidence threshold can improve unknown attack detection by forwarding only uncertain predictions to the anomaly detector, thereby balancing detection performance and computational efficiency.

In [1]:
# ============================================
# Experiment B : Hybrid Routing Optimization
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# ============================================
# Load Experiment A Results
# ============================================

analysis_df = pd.read_csv("results/rf_confidence_analysis.csv")

print("Dataset Loaded Successfully\n")

print(analysis_df.head())

print("\nShape :", analysis_df.shape)

Dataset Loaded Successfully

   Actual  Predicted  Confidence Category
0       1          0    1.000000       FN
1       1          0    1.000000       FN
2       1          0    0.999996       FN
3       1          0    0.999907       FN
4       1          0    0.999996       FN

Shape : (25122, 4)


In [3]:
# ============================================
# Candidate Thresholds
# ============================================

candidate_thresholds = {
    "50th Percentile": analysis_df["Confidence"].quantile(0.50),
    "75th Percentile": analysis_df["Confidence"].quantile(0.75),
    "90th Percentile": analysis_df["Confidence"].quantile(0.90),
    "95th Percentile": analysis_df["Confidence"].quantile(0.95),
    "99th Percentile": analysis_df["Confidence"].quantile(0.99),
}

threshold_df = pd.DataFrame(
    candidate_thresholds.items(),
    columns=["Threshold Name", "Confidence Value"]
)

print(threshold_df)

    Threshold Name  Confidence Value
0  50th Percentile          0.999952
1  75th Percentile          1.000000
2  90th Percentile          1.000000
3  95th Percentile          1.000000
4  99th Percentile          1.000000


In [4]:
analysis_df = pd.read_csv("results/rf_confidence_analysis.csv")

analysis_df.head()

,Actual,Predicted,Confidence,Category
0,1,0,1.000000,FN
1,1,0,1.000000,FN
2,1,0,0.999996,FN
3,1,0,0.999907,FN
4,1,0,0.999996,FN


In [5]:
# ============================================
# High Confidence False Negatives
# ============================================

high_conf_fn = analysis_df[
    (analysis_df["Category"] == "FN") &
    (analysis_df["Confidence"] >= 0.99)
]

print("=" * 60)
print("High Confidence False Negatives")
print("=" * 60)

print("Total FN :", len(analysis_df[analysis_df["Category"] == "FN"]))
print("High Confidence FN :", len(high_conf_fn))

percentage = (
    len(high_conf_fn)
    /
    len(analysis_df[analysis_df["Category"] == "FN"])
) * 100

print(f"Percentage : {percentage:.2f}%")

High Confidence False Negatives
Total FN : 10715
High Confidence FN : 2662
Percentage : 24.84%


In [6]:
# ============================================
# False Negative Analysis Across Thresholds
# ============================================

thresholds = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

results = []

fn_df = analysis_df[analysis_df["Category"] == "FN"]

for t in thresholds:

    count = (fn_df["Confidence"] >= t).sum()

    percentage = (count / len(fn_df)) * 100

    results.append({
        "Threshold": t,
        "FN Count": count,
        "Percentage": round(percentage, 2)
    })

threshold_results = pd.DataFrame(results)

print(threshold_results)

   Threshold  FN Count  Percentage
0       0.50     10715      100.00
1       0.60      9541       89.04
2       0.70      9205       85.91
3       0.80      8216       76.68
4       0.90      6384       59.58
5       0.95      5475       51.10
6       0.99      2662       24.84


In [7]:
# ============================================
# Cell 5 : Load Original Unknown Test Dataset
# ============================================

import pandas as pd

unknown_df = pd.read_csv("processed/unknown_test.csv")

print("Shape :", unknown_df.shape)

print("\nColumns:")
print(unknown_df.columns.tolist())

print("\nLabel Distribution:")
print(unknown_df["Label"].value_counts())

Shape : (25122, 79)

Columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CW

In [8]:
# ============================================
# Cell 6 : Attach Original Attack Labels
# ============================================

# Copy attack labels
analysis_df["Attack_Type"] = unknown_df["Label"]

print("=" * 60)
print("Attack Labels Added")
print("=" * 60)

print(analysis_df.head())

print("\nAttack Distribution:")
print(analysis_df["Attack_Type"].value_counts())

Attack Labels Added
   Actual  Predicted  Confidence Category Attack_Type
0       1          0    1.000000       FN         Bot
1       1          0    1.000000       FN         Bot
2       1          0    0.999996       FN         Bot
3       1          0    0.999907       FN         Bot
4       1          0    0.999996       FN         Bot

Attack Distribution:
Attack_Type
BENIGN              12561
DoS slowloris        5385
DoS Slowhttptest     5228
Bot                  1948
Name: count, dtype: int64


In [9]:
# ============================================
# Cell 7 : False Negatives by Attack Type
# ============================================

fn_attack = (
    analysis_df[analysis_df["Category"] == "FN"]
    .groupby("Attack_Type")
    .size()
    .reset_index(name="False_Negatives")
)

fn_attack = fn_attack.sort_values(
    by="False_Negatives",
    ascending=False
)

print("=" * 60)
print("False Negatives by Attack Type")
print("=" * 60)

print(fn_attack)

False Negatives by Attack Type
        Attack_Type  False_Negatives
1  DoS Slowhttptest             5053
2     DoS slowloris             3714
0               Bot             1948


In [10]:
# ============================================
# Cell 8 : False Negative Rate
# ============================================

attack_total = (
    unknown_df["Label"]
    .value_counts()
    .rename_axis("Attack_Type")
    .reset_index(name="Total")
)

fn_rate = attack_total.merge(
    fn_attack,
    on="Attack_Type",
    how="left"
)

fn_rate["False_Negatives"] = fn_rate["False_Negatives"].fillna(0)

fn_rate["FN_Rate (%)"] = (
    fn_rate["False_Negatives"] /
    fn_rate["Total"]
) * 100

fn_rate = fn_rate.sort_values(
    by="FN_Rate (%)",
    ascending=False
)

print(fn_rate)

        Attack_Type  Total  False_Negatives  FN_Rate (%)
3               Bot   1948           1948.0   100.000000
2  DoS Slowhttptest   5228           5053.0    96.652640
1     DoS slowloris   5385           3714.0    68.969359
0            BENIGN  12561              0.0     0.000000
